# Simple Linear Regression as a Neural Network


With one predictor variable the simple linear model predicts an outcome variable as a linear function of the predictor variable:

$$
\hat{y}_i = \hat{\beta}_0 + \hat{\beta}_1 x_{i}
$$

Where the coefficients are chosen to minimize the mean squared error loss function.

$$
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

A linear regression model is a special case of a neural network with no hidden layers, a linear activation function, and a mean squared error loss function. To be consistent with the literature on neural networks we will rename $\hat{\beta}_1$ to $w$ (weight) and $\hat{\beta}_0$ to $b$ (bias). The equation for the ANN model is
absolutely the same as for linear regression:

$$
\hat{y}_i = b + w x_{i}
$$

The usual visualization of this model is a single neuron with $p$ inputs and a single output. The neuron computes the weighted sum of its inputs and passes this value to its output nodes. For linear regression the output of the neuron is passed through an identity activation function $\text{id}(z) = z$ (i.e. unchanged) to the output node which is the predicted value $\hat{y}_i$.

In the simplest case we have one input (feature) and one output (target variable):

$$
\hat{y}_i = \text{id}(b + w x_i)
$$

![Linear Regression as a Neural Network](./figures/ann/Linear-Regression-Single-Input-Single-Output.png){style="max-width: 600px; position: relative; left: 50%; transform: translateX(-50%); margin: 1rem;"}

{{< video ./videos/SingleNeuronComputation.mp4 >}}

So if we know the values of the weight and the bias, we can compute the predicted value for any input by simply multiplying the input by the weight and adding the bias. How do we find sensible values for the weight and the bias?


## Gradient Descent

Gradient descent is an optimization algorithm that is used in machine learning to minimize a loss function by iteratively moving towards the steepest descent as defined by the negative of the gradient. In the case of
linear regression we can minimize the mean squared error directly using calculus (least squares solution). For more complex models there are often no closed form solutions and we have to resort to numerical optimization methods such as gradient descent.

To illustrate the algorithm we will first consider a very simple function of a single variable $w$:

$$
L(w) = (w - 1)^2 + 0.5
$$

We can directly see that the minimum of the function is 0.5 at $w=1$ (because the squared term is always non-negative and is zero at $w=1$). In the case of more 
complicated functions it may not be as easy to see the minimum directly so we can find it by setting the derivative to zero and solving for $w$:

$$
\frac{dL}{dw} = 2(w - 1) = 0
$$

Obviously this also gives us $w=1$ as the minimum. The sufficient condition for a minimum is that the second derivative is positive which it is in this case (and we can also see this from the graph of the function).

$$
\frac{d^2L}{dw^2} = 2 > 0
$$

Let's now apply the gradient descent algorithm. It starts by picking a some initial value for $w$ (let's say $w_0 = -2$ as in the video below) and computing the derivative of the loss function at that point:

$$
\frac{dL}{dw}\bigg|_{w=w_0} = 2(w_0 - 1) = 2(-2 - 1) = -6
$$

The derivative is non-zero so we are not at the minimum yet. The derivative is negative which means that the function will decrease (what we want) if we increase $w$. The question now is by how much should we change $w$? The update rule for gradient descent is to add the negative of the derivative multiplied by a small constant called the learning rate $\eta$:

$$
w_1 = w_0 - \eta \frac{dL}{dw}\big|_{w=w_0}
$$

Taking a learning rate of $\eta = 0.3$ we get:

$$
w_1 = -2 - 0.3 \cdot (-6) = -0.2
$$

The derivative at this new point is:

$$
\frac{dL}{dw}\bigg|_{w=w_1} = 2(w_1 - 1) = 2(-0.2 - 1) = -2.4
$$

so the function is still decreasing if we increase $w$. We can repeat the update step:

$$
w_2 = w_1 - \eta \frac{dL}{dw}\big|_{w=w_1} = -0.2 - 0.3 \cdot (-2.4) = 0.52
$$

We can continue this process iteratively until the derivative is close to zero (indicating that we are close to the minimum). The video below shows how the value of $w$ converges to the minimum at $w=1$ over several iterations of gradient descent.

The learning rate $\eta$ controls how big steps we take towards the minimum. If $\eta$ is too large we may overshoot the minimum and diverge instead of converging. If $\eta$ is too small we will walk towards the minimum with very small steps so the algorithm will take a long time to converge. Usually we have to experiment with different values of $\eta$ to find a good one for a particular problem, a process called hyperparameter tuning. Usually values between 0.001 and 0.1 are a good starting point. Look at the animation below and experiment with different values of $\eta$. (We use the larger value $\eta = 0.3$ in this toy example only because the function is a simple quadratic.)

{{< video ./videos/GradientDescentQuadratic.mp4 >}}


The algorithm is not guaranteed to find the global minimum of the loss function when there are multiple local minima. However, in practice it often finds a good enough solution for many machine learning problems.

{{< video ./videos/GradientDescentPolynomial.mp4 >}}


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [ ]:
# Generate synthetic data to setup a linear regression problem as an example

# Number of observations
n = 100

# Number of predictors
p = 1

np.random.seed(23)

# A n x p matrix of predictors

X = np.random.randn(n, p)

# Random noise 
e = np.random.randn(n)

# True coefficient of x
beta = np.array([2])

# Response variable
y = 2 + X @ beta + e

# Standardize y for better numerical stability
# x is drawn from N(0,1), so it is approximately standardized  # [XXX check]

y = (y - np.mean(y)) / np.std(y)

In [ ]:
# Fit linear regression so that 
# we can compare with the network result later

linreg = LinearRegression()
linreg.fit(X, y)

# Note that the coefficinets are not close to the true values
# in the simulation (b = 2 and w=2) because of the standardization of y. 
# The coefficients are now in standardized units.

print("Intercept:", linreg.intercept_)
print("Coefficient:", linreg.coef_)


In [ ]:
# Plot the data and the fitted regression line

plt.scatter(X, y)
plt.plot(X, linreg.predict(X), color='red', label = r'$\hat{y}$')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()


## Gradient Descent for the Simple Linear Regression

{{< video ./videos/GradientDescentLinearRegressionTwoParams.mp4 >}}


We can use gradient descent to find the optimal values of the weight and the bias that minimize the loss function. The loss function for linear regression is the mean squared error (MSE). For a single observation (sample) the squared error loss is:

$$
L_i = (\text{actual} - \text{predicted})^2 = (y_i - \hat{y}_i)^2, \qquad \hat{y}_i = w x_i + b
$$

The gradient descent algorithm iteratively updates the weight and the bias in the direction of the negative gradient of the loss function with respect to the weight and the bias.

$$
\begin{align*}
w^{\text{new}} & = w^{\text{old}} - \eta \frac{\partial L_i}{\partial w} \\
b^{\text{new}} & = b^{\text{old}} - \eta \frac{\partial L_i}{\partial b}
\end{align*}
$$

where $\eta$ is the learning rate, a hyperparameter that controls the step size of the updates. To actually perform the updates, we need to compute the gradients of the loss function with respect to the weight and the bias. This is quite straightforward here (the factor $-2$ comes from differentiating the square: $\frac{\partial}{\partial \hat{y}} (y - \hat{y})^2 = -2 (y - \hat{y})$):


$$
\begin{align*}
\frac{\partial L_i}{\partial w} & = \frac{\partial }{\partial w} (y_i - w x_i - b)^2 \\
                                  & = -2 (y_i - \hat{y}_i) \frac{\partial }{\partial w} \hat{y}_i \\
                                  & = -2 (y_i - \hat{y}_i) x_{i}
\end{align*}
$$

$$
\begin{align*}
\frac{\partial L_i}{\partial b} & = \frac{\partial }{\partial b} (y_i - w x_i - b)^2 \\
                                  & = -2 (y_i - \hat{y}_i) \frac{\partial }{\partial b} \hat{y}_i \\
                                  & = -2 (y_i - \hat{y}_i)
\end{align*}
$$

It is useful to also compute the update equations not just for a single observation, but for a subset (batch) of the observations.



The MSE loss for the $n$ observations in the data (or in a batch of size $n$) is:

$$
L = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

Exactly as before, we differentiate the loss with respect to the weight and the bias to get the update equations for the batch:

$$
\begin{align*}
\frac{\partial L}{\partial w} & = \frac{1}{n} \sum_{i=1}^{n} \frac{\partial }{\partial w} (y_i - \hat{y}_i)^2 \\
                                    & = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i) \frac{\partial }{\partial w} \hat{y}_i \\
                                    & = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i) x_{i}                     
\end{align*}
$$

$$
\frac{\partial L}{\partial b} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)
$$

It is also convenient to express these equations in vectorized form. It has the advantage that it maps nicely to matrix operations that we can express directly using `numpy`. Another advantage is that it generalizes easily to multiple input features and multiple output targets. Furthermore, vectorized operations are usually more computationally efficient. 

The sum involved in the gradient computation above is just the dot product of the column vectors of the errors $(y - \hat{y})$ and the inputs $x$ (for all observations):

$$
x = \begin{bmatrix}
x_{1} \\
x_{2} \\
\vdots \\
x_{n}
\end{bmatrix}
\quad
(y - \hat{y}) = \begin{bmatrix}
y_{1} - \hat{y}_{1} \\
y_{2} - \hat{y}_{2} \\
\vdots \\
y_{n} - \hat{y}_{n}
\end{bmatrix}
$$

$$
\begin{align*}
\frac{\partial L}{\partial w} & = -\frac{2}{n} x^\top (y - \hat{y}) \\
\frac{\partial L}{\partial b} & = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)
\end{align*}
$$

With $p > 1$ predictors, $x$ is replaced by the $n \times p$ matrix $X$ and the gradient with respect to the $p \times 1$ weight vector is $-\frac{2}{n} X^\top (y - \hat{y})$.

## Simple Linear Regression as an ANN with Gradient Descent

First it is instructive to implement the forward pass of the ANN for simple linear regression manually (without using `sklearn` or `statsmodels` or an AI assistant to write the code). This helps enormously with understanding the mechanics.

In [ ]:
# Hyperparameters

# How many times to go through the data
epochs = 200

# The learning rate
lr = 0.05

# First we initialize the parameters to 
# small random values  # [XXX check]

w = np.random.randn(p, 1)  # shape (p, 1) so that X @ w works for any p 
b = np.random.randn(1)

print("Initial weights:", w)
print("Initial bias:", b)

losses = []

# Training loop
for epoch in range(epochs):
    # Forward pass
    y_pred = X @ w + b
    
    # Compute loss (MSE)
    loss = np.mean((y.reshape(-1, 1) - y_pred) ** 2)
    losses.append(loss)
    
    # Backward pass (compute gradients)
    errors = y.reshape(-1, 1) - y_pred

    gradient_w = -2 * X.T @ errors / n
    gradient_b = -2 * errors.mean()
    
    # Update parameters
    w = w - lr * gradient_w
    b = b - lr * gradient_b

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1} Updated weights:", w)  # [XXX check] was "Iteration"
        print(f"Epoch {epoch + 1} Updated bias:", b)

# Print final parameters
print("Weight:", w.item())
print("Bias:", b.item())

# Plot training loss
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs (NumPy)')
plt.show()

# Plot predictions
y_pred_final = X @ w + b

plt.scatter(X, y, label='Data')
plt.plot(X, y_pred_final, color='red', label='Prediction')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()

## Implementation in PyTorch

It is also instructive to implement it in PyTorch as this is the simplest example of a neural network. Compare the results, all three implementations (`sklearn`, `numpy` and `pytorch`) should arrive at the same values for $b and w$.


In [ ]:
import torch

# Convert the existing NumPy data to PyTorch tensors
X_torch = torch.from_numpy(X).float()
y_torch = torch.from_numpy(y.reshape(-1, 1)).float()

# Define a linear regression model: one output and no hidden layers
linear_model_torch = nn.Linear(p, 1)

criterion_torch = nn.MSELoss()
optimizer_torch = torch.optim.SGD(linear_model_torch.parameters(), lr=0.05)

epochs_torch = 1000
losses_torch = []

for epoch in range(epochs_torch):
    # Forward pass and loss
    y_pred_torch = linear_model_torch(X_torch)
    loss_torch = criterion_torch(y_pred_torch, y_torch)
    losses_torch.append(loss_torch.item())

    # Compute gradients and update parameters
    optimizer_torch.zero_grad()
    loss_torch.backward()
    optimizer_torch.step()

print("Intercept:", linear_model_torch.bias.item())
print("Coefficient:", linear_model_torch.weight.detach().numpy().ravel())
print("Final MSE:", losses_torch[-1])

plt.plot(losses_torch)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("PyTorch Linear Regression Training Loss")
plt.show()

## What's next

A single neuron with a linear activation is just a linear regression. In the next chapter we add more input features and then a hidden layer with a non-linear activation function: [Multiple regression and hidden layers](03-04-ANN-Multiple-Regression.ipynb).